# Лабораторна робота №2 — Частина 1
## Наука про дані: підготовчий етап
### Завантаження та обробка VHI-даних по регіонах України

## 0. Імпорт бібліотек

In [ ]:
import os
import re
import urllib.request
import pandas as pd
from datetime import datetime

## 1. Створення директорії та завантаження VHI-файлів

Для кожної з адміністративних одиниць України завантажити (urllib) текстові структуровані файли, що містять значення VHI-індексу. При збереженні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного завантаження та колізії даних.

In [ ]:
DATA_DIR = "vhi_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = (
    "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php"
    "?country=UKR&provinceID={pid}&year1=1981&year2=2024&type=Mean"
)

def already_downloaded(province_id: int) -> bool:
    """Повертає True, якщо файл для даної області вже існує у DATA_DIR."""
    pattern = re.compile(rf"^vhi_province_{province_id}_\d{{8}}_\d{{6}}\.csv$")
    return any(pattern.match(f) for f in os.listdir(DATA_DIR))


def download_vhi(province_id: int) -> str | None:
    """Завантажує VHI-файл для вказаної області. Повертає шлях до файлу або None."""
    if already_downloaded(province_id):
        print(f"Province {province_id:2d}: вже завантажено, пропускаємо.")
        return None

    url = BASE_URL.format(pid=province_id)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"vhi_province_{province_id}_{timestamp}.csv"
    filepath = os.path.join(DATA_DIR, filename)

    try:
        urllib.request.urlretrieve(url, filepath)
        print(f"Province {province_id:2d}: збережено → {filename}")
        return filepath
    except Exception as e:
        print(f"Province {province_id:2d}: помилка — {e}")
        return None


# Завантажуємо всі 27 областей (provinceID 1..27; 0 — Україна загалом, не завантажуємо)
for pid in range(1, 28):
    download_vhi(pid)

print("\nГотово. Файли у директорії:", os.listdir(DATA_DIR)[:5], "...")

## 2. Зчитування файлів у DataFrame та Data Cleaning

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області.

In [ ]:
# Маппінг NOAA province_id → назва у NOAA (англ. абетка)
NOAA_ID_TO_NAME = {
    1:  "Cherkasy",
    2:  "Chernihiv",
    3:  "Chernivtsi",
    4:  "Crimea",
    5:  "Dnipropetrovsk",
    6:  "Donetsk",
    7:  "Ivano-Frankivsk",
    8:  "Kharkiv",
    9:  "Kherson",
    10: "Khmelnytskyi",
    11: "Kirovohrad",
    12: "Kyiv",
    13: "Kyiv City",
    14: "Luhansk",
    15: "Lviv",
    16: "Mykolaiv",
    17: "Odessa",
    18: "Poltava",
    19: "Rivne",
    20: "Sumy",
    21: "Ternopil",
    22: "Vinnytsia",
    23: "Volyn",
    24: "Zakarpattia",
    25: "Zaporizhzhia",
    26: "Zhytomyr",
    27: "Sevastopol",
}


def read_vhi_file(filepath: str, noaa_id: int) -> pd.DataFrame:
    """Читає один VHI CSV-файл, очищає та повертає DataFrame."""
    with open(filepath, "r", encoding="utf-8") as f:
        raw = f.read()

    # Видаляємо HTML/XML-заголовки та порожні рядки
    lines = []
    for line in raw.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("<"):
            continue
        lines.append(stripped)

    from io import StringIO
    cleaned = "\n".join(lines)
    df = pd.read_csv(
        StringIO(cleaned),
        header=0,
        skipinitialspace=True,
        on_bad_lines="skip",
    )

    # Нормалізуємо назви колонок
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # Залишаємо тільки потрібні колонки
    expected = ["year", "week", "smi", "vci", "tci", "vhi", "empty"]
    df = df[[c for c in df.columns if c in expected or "vhi" in c or "year" in c or "week" in c]]

    # Видаляємо рядки, де VHI = -1 (пропуск у даних NOAA)
    if "vhi" in df.columns:
        df = df[df["vhi"] != -1]

    # Заповнюємо NaN числових колонок медіаною
    num_cols = df.select_dtypes(include="number").columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    # Додаємо мета-стовпці
    df["noaa_province_id"] = noaa_id
    df["province_name_en"] = NOAA_ID_TO_NAME.get(noaa_id, "Unknown")

    df = df.reset_index(drop=True)
    return df


def load_all_vhi() -> pd.DataFrame:
    """Читає всі файли з DATA_DIR та об'єднує в один DataFrame."""
    dfs = []
    pattern = re.compile(r"vhi_province_(\d+)_")
    for fname in sorted(os.listdir(DATA_DIR)):
        m = pattern.match(fname)
        if not m:
            continue
        noaa_id = int(m.group(1))
        path = os.path.join(DATA_DIR, fname)
        try:
            df = read_vhi_file(path, noaa_id)
            dfs.append(df)
        except Exception as e:
            print(f"Помилка при читанні {fname}: {e}")

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


vhi_df = load_all_vhi()
print("Розмір датасету:", vhi_df.shape)
display(vhi_df.head(10))

## 3. Переіндексація областей (NOAA → українська абетка)

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 — Cherkasy), потрібно замінити індекси так, щоб області індексувалися за українською абеткою (1 область — Вінницька).

In [ ]:
# Відповідність: українська назва → noaa_id  (за алфавітом укр. мови)
UKR_ALPHA_ORDER = [
    (1,  "Вінницька",       22),
    (2,  "Волинська",       23),
    (3,  "Дніпропетровська",5),
    (4,  "Донецька",        6),
    (5,  "Житомирська",     26),
    (6,  "Закарпатська",    24),
    (7,  "Запорізька",      25),
    (8,  "Івано-Франківська",7),
    (9,  "Київська",        12),
    (10, "Кіровоградська",  11),
    (11, "Луганська",       14),
    (12, "Львівська",       15),
    (13, "Миколаївська",    16),
    (14, "Одеська",         17),
    (15, "Полтавська",      18),
    (16, "Рівненська",      19),
    (17, "Сумська",         20),
    (18, "Тернопільська",   21),
    (19, "Харківська",      8),
    (20, "Херсонська",      9),
    (21, "Хмельницька",     10),
    (22, "Черкаська",       1),
    (23, "Чернівецька",     3),
    (24, "Чернігівська",    2),
    (25, "Крим",            4),
    (26, "м. Київ",         13),
    (27, "м. Севастополь",  27),
]

NOAA_TO_UKR_IDX  = {noaa: ukr  for ukr, _, noaa in UKR_ALPHA_ORDER}
NOAA_TO_UKR_NAME = {noaa: name for _,  name, noaa in UKR_ALPHA_ORDER}


def reindex_provinces(df: pd.DataFrame) -> pd.DataFrame:
    """Додає колонки ukr_province_id та province_name_ukr за українською абеткою."""
    df = df.copy()
    df["ukr_province_id"]  = df["noaa_province_id"].map(NOAA_TO_UKR_IDX)
    df["province_name_ukr"] = df["noaa_province_id"].map(NOAA_TO_UKR_NAME)
    return df


vhi_df = reindex_provinces(vhi_df)
print("Колонки після переіндексації:", vhi_df.columns.tolist())
display(vhi_df[["noaa_province_id", "province_name_en", "ukr_province_id", "province_name_ukr"]].drop_duplicates().sort_values("ukr_province_id"))

## 4. Вибірки

Реалізувати процедури для формування вибірок.

### 4.1 Ряд VHI для області за вказаний рік

In [ ]:
def vhi_by_province_year(df: pd.DataFrame, ukr_id: int, year: int) -> pd.DataFrame:
    """
    Повертає ряд VHI для вказаної області (ukr_province_id) за вказаний рік.
    """
    result = df[(df["ukr_province_id"] == ukr_id) & (df["year"] == year)][
        ["year", "week", "vhi", "province_name_ukr"]
    ].reset_index(drop=True)
    print(f"VHI для '{result['province_name_ukr'].iloc[0] if not result.empty else ukr_id}', {year} рік:")
    return result


# Приклад: Харківська (id=19) за 2020 рік
display(vhi_by_province_year(vhi_df, ukr_id=19, year=2020))

### 4.2 Ряд VHI за вказаний діапазон років для вказаних областей

In [ ]:
def vhi_range(df: pd.DataFrame, ukr_ids: list[int], year_from: int, year_to: int) -> pd.DataFrame:
    """
    Повертає VHI для переліку областей (ukr_province_id) за діапазон років.
    """
    result = df[
        df["ukr_province_id"].isin(ukr_ids)
        & df["year"].between(year_from, year_to)
    ][["year", "week", "vhi", "ukr_province_id", "province_name_ukr"]].reset_index(drop=True)
    print(f"VHI для областей {ukr_ids}, {year_from}–{year_to}: {len(result)} записів")
    return result


# Приклад: Харківська (19) та Полтавська (15), 2015–2020
display(vhi_range(vhi_df, ukr_ids=[19, 15], year_from=2015, year_to=2020))

### 4.3 Пошук екстремумів (min, max), середнього та медіани

In [ ]:
def vhi_stats(df: pd.DataFrame, ukr_ids: list[int], year_from: int, year_to: int) -> pd.DataFrame:
    """
    Обчислює min, max, mean та median VHI для вказаних областей і діапазону років.
    """
    subset = df[
        df["ukr_province_id"].isin(ukr_ids)
        & df["year"].between(year_from, year_to)
    ]
    stats = (
        subset.groupby(["ukr_province_id", "province_name_ukr"])["vhi"]
        .agg(min="min", max="max", mean="mean", median="median")
        .reset_index()
    )
    print(f"Статистика VHI для областей {ukr_ids}, {year_from}–{year_to}:")
    return stats


# Приклад
display(vhi_stats(vhi_df, ukr_ids=[19, 15, 3], year_from=2000, year_to=2024))

### 4.4 Додаткова вибірка: роки з екстремальною посухою (VHI < 15) по всіх областях

In [ ]:
def extreme_drought_years(df: pd.DataFrame, vhi_threshold: float = 15.0) -> pd.DataFrame:
    """
    Повертає записи, де VHI нижчий за поріг (екстремальна посуха).
    Групує по роках та областях, показує кількість тижнів з посухою.
    """
    drought = df[df["vhi"] < vhi_threshold]
    result = (
        drought.groupby(["year", "ukr_province_id", "province_name_ukr"])
        .size()
        .reset_index(name="drought_weeks")
        .sort_values("drought_weeks", ascending=False)
        .reset_index(drop=True)
    )
    print(f"Записи з VHI < {vhi_threshold} (екстремальна посуха): {len(drought)} рядків")
    return result


display(extreme_drought_years(vhi_df).head(20))